# Late Arriving Reprocess

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/06_advanced_workflows/late_arriving_reprocess/demo_self_healing.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/06_advanced_workflows/late_arriving_reprocess/demo_self_healing.ipynb)

## Business Scenario

Sales data arrives days late, after daily aggregates have already been computed. You need your Gold metrics to heal automatically when late records show up.

## Value Proposition

- Automatically reprocess impacted partitions
- Preserve historical accuracy in Gold metrics
- Avoid full reloads for late data

---

## Goals

1. Run an initial batch
2. Ingest late records
3. Observe self-healing aggregates


## 🚀 Step 1: Initial Load (Batch 1)

We process the first batch of events and create our Gold Product.

In [ ]:
from lakelogic import DataProcessor
import polars as pl
import os
import shutil

# Clean start
for d in ['data/silver', 'data/gold', 'logs']: 
    if os.path.exists(d): shutil.rmtree(d)

silver_proc = DataProcessor(contract="silver_contract.yaml")
gold_proc = DataProcessor(contract="gold_contract.yaml")

print("📥 Processing Batch 1 (Initial Data)...")
silver_proc.run("data/raw_sales_batch_1.csv")
gold_proc.run_source() # Gold reads from the silver folder defined in its contract

df_gold = pl.read_parquet("data/gold/revenue")
print("\n📊 Initial Gold Revenue Product:")
print(df_gold.sort("event_date"))

## 🩹 Step 2: The Late Arrival (Batch 2)

Now a record for **2024-02-05** (Monday) arrives late. Notice that we don't have to change any code—we just run the same contracts.

In [ ]:
print("📥 Processing Batch 2 (Late Arrival for Feb 5th)...")
silver_proc.run("data/raw_sales_batch_2_LATE.csv")

print("🔄 Gold layer detecting changes and healing product...")
gold_proc.run_source()

df_gold_healed = pl.read_parquet("data/gold/revenue")
print("\n✨ Healed Gold Revenue Product:")
print(df_gold_healed.sort("event_date"))

## 🧐 Why did this work?

1.  **Lineage Watermark**: The late record in Silver was tagged with a `_lakelogic_processed_at` time of **Now**.
2.  **Incremental Selection**: Gold saw that new records were added to Silver since the last run.
3.  **Partition Healing**: Because we used `reprocess_policy: overwrite_partition_safe`, LakeLogic calculated the new aggregate for Feb 5th and replaced the old file in the `event_date=2024-02-05` directory.

This is the heart of **Data as a Product**: No manual maintenance, just a self-healing system.